# 04 — Validation

QA pass over the pipeline built in notebooks 01-03: real cross-validation math (SMAP vs
ISMN), unit-convention sanity checks that fail loudly (`CLAUDE.md` rule 5: explicit units;
rule 6: safe default is closed — a validator that silently passes bad data is worse than none),
and a `data_quality` report mirroring the enums in `docs/schemas.md`.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(99)
WINDOW_DAYS = 30
dates = pd.date_range("2025-06-01", periods=WINDOW_DAYS, freq="D", tz="UTC")

def seasonal_signal(n, base, amplitude, noise_std, period=365, phase=150, rng=rng):
    t = np.arange(n)
    signal = base + amplitude * np.sin(2 * np.pi * (t + phase) / period)
    return signal + rng.normal(0, noise_std, size=n)


## SMAP vs ISMN cross-validation (real math, simulated inputs)

In [2]:
smap_theta = np.clip(seasonal_signal(WINDOW_DAYS, base=0.22, amplitude=0.08, noise_std=0.015), 0.0, 1.0)
ismn_theta = np.clip(smap_theta + rng.normal(0.01, 0.02, size=WINDOW_DAYS), 0.0, 1.0)

def cross_validate(reference, candidate):
    reference = np.asarray(reference, dtype=float)
    candidate = np.asarray(candidate, dtype=float)
    mask = ~(np.isnan(reference) | np.isnan(candidate))
    ref, cand = reference[mask], candidate[mask]
    if ref.size < 2:
        raise ValueError("Not enough paired observations to cross-validate")
    r = np.corrcoef(ref, cand)[0, 1]
    rmse = np.sqrt(np.mean((cand - ref) ** 2))
    bias = np.mean(cand - ref)
    return {"n_pairs": int(ref.size), "pearson_r": r, "rmse_m3m3": rmse, "bias_m3m3": bias}

validation_result = cross_validate(reference=ismn_theta, candidate=smap_theta)
validation_result


{'n_pairs': 30,
 'pearson_r': np.float64(0.7716745577866494),
 'rmse_m3m3': np.float64(0.022859950163058203),
 'bias_m3m3': np.float64(-0.013662464273809101)}

## Unit-convention sanity checks (fail loudly, per CLAUDE.md rules 5 and 6)

CLAUDE.md rule 5: soil moisture is always volumetric (m3/m3); water depth is mm; time is UTC.
CLAUDE.md rule 6: on any path that can affect an irrigation decision, uncertainty must resolve
to "do not irrigate" — a validator that cannot distinguish good data from bad is worse than no
validator. These checks `raise`, they do not warn-and-continue.


In [3]:
def assert_soil_moisture_valid(theta_m3m3, name="theta"):
    theta_m3m3 = np.asarray(theta_m3m3, dtype=float)
    finite = theta_m3m3[~np.isnan(theta_m3m3)]
    if finite.size == 0:
        raise ValueError(f"{name}: no finite values to validate")
    if (finite < 0).any() or (finite > 1).any():
        bad = finite[(finite < 0) | (finite > 1)]
        raise ValueError(f"{name}: {bad.size} values outside physical range [0, 1] m3/m3: {bad[:5]}")

def assert_depth_non_negative(depth_mm, name="depth_mm"):
    depth_mm = np.asarray(depth_mm, dtype=float)
    finite = depth_mm[~np.isnan(depth_mm)]
    if (finite < 0).any():
        bad = finite[finite < 0]
        raise ValueError(f"{name}: {bad.size} negative depth values (mm must be >= 0): {bad[:5]}")

def assert_timestamps_utc_aware(timestamps, name="timestamps"):
    ts = pd.DatetimeIndex(timestamps)
    if ts.tz is None:
        raise ValueError(f"{name}: timestamps are timezone-naive; CLAUDE.md rule 5 requires explicit UTC")
    if str(ts.tz) != "UTC":
        raise ValueError(f"{name}: timestamps are tz-aware but not UTC ({ts.tz}); convert explicitly at the edge layer")

# These must all pass on well-formed simulated data
assert_soil_moisture_valid(smap_theta, "smap_theta")
assert_soil_moisture_valid(ismn_theta, "ismn_theta")
assert_timestamps_utc_aware(dates, "dates")
assert_depth_non_negative(np.abs(rng.normal(5, 2, WINDOW_DAYS)), "measured_depth_mm")
print("All unit-convention checks passed on well-formed data.")


All unit-convention checks passed on well-formed data.


In [4]:
# Demonstrate that the checks actually fail loudly on bad data — this is the point of rule 6.
bad_theta = np.array([0.2, 0.3, 1.4, -0.1, 0.25])  # 1.4 and -0.1 are physically impossible

try:
    assert_soil_moisture_valid(bad_theta, "bad_theta")
    raise AssertionError("validator should have raised on out-of-range soil moisture and did not")
except ValueError as e:
    print(f"Correctly rejected bad data: {e}")


Correctly rejected bad data: bad_theta: 2 values outside physical range [0, 1] m3/m3: [ 1.4 -0.1]


## Data quality report

Mirrors the `data_quality` enum on `IRRIGATION_LOG` (`ok`, `meter_absent`, `meter_suspect`,
`clock_uncertain`) from `docs/schemas.md`, applied here to the simulated pipeline outputs as a
structural placeholder for the real per-record QA the ETL layer will emit.


In [5]:
from enum import Enum

class DataQuality(str, Enum):
    OK = "ok"
    METER_ABSENT = "meter_absent"
    METER_SUSPECT = "meter_suspect"
    CLOCK_UNCERTAIN = "clock_uncertain"

def build_quality_report(smap_theta, ismn_theta, dates, cv_result):
    flags = []
    if cv_result["pearson_r"] < 0.5:
        flags.append(DataQuality.METER_SUSPECT)  # placeholder mapping: poor SMAP/ISMN agreement
    if not pd.DatetimeIndex(dates).tz:
        flags.append(DataQuality.CLOCK_UNCERTAIN)
    if len(flags) == 0:
        flags.append(DataQuality.OK)
    return {
        "n_records": len(dates),
        "flags": [f.value for f in flags],
        "smap_ismn_pearson_r": cv_result["pearson_r"],
        "smap_ismn_rmse_m3m3": cv_result["rmse_m3m3"],
    }

quality_report = build_quality_report(smap_theta, ismn_theta, dates, validation_result)
quality_report


{'n_records': 30,
 'flags': ['ok'],
 'smap_ismn_pearson_r': np.float64(0.7716745577866494),
 'smap_ismn_rmse_m3m3': np.float64(0.022859950163058203)}

## Summary

This notebook's cross-validation and unit-check *logic* is real and reusable once live SMAP
and ISMN feeds exist — only the input arrays are synthetic. The loud-failure pattern
(`raise`, not warn-and-continue) is deliberate and should carry over directly into the
production ETL validation step in `packages/etl/`, per CLAUDE.md rule 6: on any actuation-
adjacent path, uncertain or invalid data must resolve to "do not irrigate", never to a
best-guess default.
